# Count number of each nucleotide within reads and the number of each nucleotide at each position for distinct inserts

## Import Libraries

In [1]:
import pandas as pd
import re
from Bio.SeqIO.QualityIO import FastqGeneralIterator
import itertools as it
import sys

## Define Paths to Files

In [2]:
annotations_file = "../annotations/sample_annotations.csv"
linked_barcode_recorder_file = f"../data/linked_barcodes/i84p1.csv"
output_file = f"../data/summary_stats/i84p1.csv"

target_edit_loc = [1,2,4,7,9,12,13,15]
target_sequence = 'ATTATGGTGTAATTCTA'
target_5_var_loc = [8,10,11,14,16]
target_3_var_loc = [0,3,5,6,8]

# linked_barcode_recorder_file = sys.argv[1]
# fastq_file=sys.argv[2]
# fastq_umi=sys.argv[3]
# output_file= sys.argv[4]
# barcode = sys.argv[5]

## Read in Barcode Recorder File

In [3]:
edit_data = pd.read_csv(linked_barcode_recorder_file)

## Iterate through reads and parse the summary stats we want to collect

In [4]:
count_table = dict()

for _, row in edit_data.iterrows():
    barcode = row['barcode1']
    target = row['recorder']

    if barcode not in count_table:
        count_table[barcode] = {'read_counts': 0}


    count_table[barcode]['read_counts'] += 1

    variable_target = ''.join(target[loc] for loc in target_edit_loc)

    for num,nt in it.product(range(len(variable_target) + 1), ['A','G','T','C','N']):
        key = f'num_{num}_{nt}'
        if  key not in count_table[barcode]:
            count_table[barcode][key] = 0
        
    for nt in ['A','G','T','C','N']:
        count = variable_target.count(nt)
        count_table[barcode][f"num_{count}_{nt}"] += 1


    for num,nt in it.product(range(len(variable_target)), ['A','G','T','C','N']):
        key = f'pos_{num}_{nt}'
        if  key not in count_table[barcode]:
            count_table[barcode][key] = 0

    for pos,char in enumerate(variable_target):
        count_table[barcode][f'pos_{pos}_{char}'] += 1


## Write summary stats to output file

In [5]:
count_table_df = pd.DataFrame.from_dict(count_table, orient='index')
count_table_df.to_csv(output_file, index_label='barcode')